# Uber Eats Mx Churning Prediction

In [8]:
import os
import pandas as pd

# 读取 V2 版本 Excel 文件
xlsx_path = '/workspaces/ML_Uber_Eat_Churn_Prediction_Student/uber_eats_merchant_churn_synthetic_data_v2.xlsx'
if not os.path.exists(xlsx_path):
    xlsx_path = '/workspaces/ML_Project_Uber_Eat_Churn_Prediction/uber_eats_merchant_churn_synthetic_data_v2.xlsx'

print('Data file:', xlsx_path)
xl = pd.ExcelFile(xlsx_path)
print('Sheets:', xl.sheet_names)

# 读取核心表
merchant_dim_df = pd.read_excel(xlsx_path, sheet_name='merchant_dim')
merchant_daily_df = pd.read_excel(xlsx_path, sheet_name='merchant_daily_sample')
orders_fact_df = pd.read_excel(xlsx_path, sheet_name='orders_sample')
support_fact_df = pd.read_excel(xlsx_path, sheet_name='support_fact')
promotion_fact_df = pd.read_excel(xlsx_path, sheet_name='promotion_fact')

# 先看关键表结构
for name, df in {
    'merchant_dim': merchant_dim_df,
    'merchant_daily_sample': merchant_daily_df,
    'orders_sample': orders_fact_df,
    'support_fact': support_fact_df,
    'promotion_fact': promotion_fact_df,
}.items():
    display(df.head(3))
    print(f'\n--- {name} ---')
    print(df.shape)

Data file: /workspaces/ML_Uber_Eat_Churn_Prediction_Student/uber_eats_merchant_churn_synthetic_data_v2.xlsx
Sheets: ['cohort_summary', 'README', 'missingness_profile', 'signal_dictionary', 'data_quality_plan', 'feature_mart_dirty', 'merchant_dim', 'merchant_daily_sample', 'orders_sample', 'support_fact', 'promotion_fact']


,merchant_id,merchant_name,city,cuisine_type,segment,join_date,menu_ready,competitor_presence,profile_avg_rating
0,M00001,Merchant 00001,San Francisco,Chinese,SMB,2021-09-25,1,0,4.12
1,M00002,Merchant 00002,Berkeley,Thai,SMB,2023-12-12,1,1,4.38
2,M00003,Merchant 00003,Oakland,Japanese,SMB,2023-03-28,1,1,4.02



--- merchant_dim ---
(500, 9)


,merchant_id,date,is_open,scheduled_hours,actual_open_hours,business_hours_consistency,temporary_closure_flag,orders_created,accepted_orders,cancelled_orders,merchant_delay_orders,avg_prep_minutes
0,M00001,2026-02-07,1,14,-3.00,0.9092,0,2,2,0,0,13.57
1,M00001,2026-02-08,1,14,11.76,0.8400,0,0,0,0,0,NaN
2,M00001,2026-02-28,1,10,8.10,0.8102,0,1,1,0,0,19.59



--- merchant_daily_sample ---
(6000, 12)


...


--- orders_sample ---
(8000, 25)


,ticket_id,merchant_id,created_at,issue_type,resolved_flag,resolved_at,resolution_hours,merchant_satisfaction_score
0,T0000001,M00001,2026-02-08 20:00:00,Account,1,2026-02-09 06:40:21.251468617,10.67,3.0
1,T0000002,M00001,2026-02-14 13:00:00,Technical,1,2026-02-16 02:03:22.514270746,37.06,4.0
2,T0000003,M00001,2026-04-10 09:00:00,Order Issue,1,2026-04-10 14:33:29.760068863,5.56,3.0



--- support_fact ---
(1363, 8)


,promotion_id,merchant_id,promotion_type,start_date,end_date,promo_spend,promotion_return,promotion_roi
0,P0000001,M00001,Sponsored Listing,2026-05-10,2026-05-18,47.83,118.73,1.4823
1,P0000002,M00002,Free Delivery,2026-03-18,2026-03-24,124.95,314.38,1.5161
2,P0000003,M00002,Free Delivery,2026-04-08,2026-04-16,69.31,146.66,1.1160



--- promotion_fact ---
(697, 8)


In [ ]:
## Collect data

### Step 1：用 SQL 从原始表中抽取特征

先把数据整理成一个“merchant-level 的特征表”。

核心思路：
- 先从 `merchant_dim` 取商家基础信息
- 再从 `merchant_daily_sample` 计算运营活跃度
- 再从 `orders_sample` 计算 GMV、订单数、新客比、退款率等
- 再从 `support_fact` 和 `promotion_fact` 计算支持和营销信号
- 最后，将这些结果按 `merchant_id` 汇总到一张特征表

这一步就是建模前最重要的 SQL 聚合步骤。

## 订单表现

In [ ]:
# 安装 pandasql（如果环境中还没有）
%pip install pandasql -q

In [ ]:
from pandasql import sqldf

# 先把表注册成 SQL 表
# 注意：这里我们并不是直接从 feature_mart 读，而是从原始事实表逐步抽取特征
merchant_dim_df = pd.read_excel(xlsx_path, sheet_name='merchant_dim')
merchant_daily_df = pd.read_excel(xlsx_path, sheet_name='merchant_daily_sample')
orders_fact_df = pd.read_excel(xlsx_path, sheet_name='orders_sample')
support_fact_df = pd.read_excel(xlsx_path, sheet_name='support_fact')
promotion_fact_df = pd.read_excel(xlsx_path, sheet_name='promotion_fact')

# 示例：只抽取最近 28 天的订单表现指标
observation_date = '2026-06-30'

sql_order_features = '''
WITH params AS (
    SELECT DATE('2026-06-30') AS obs_date
),

)
SELECT *
FROM order_metrics
ORDER BY merchant_id;
'''

# 执行 SQL
order_feature_df = sqldf(sql_order_features, globals())
print('订单特征示例：')
display(order_feature_df.head())
